***
## Setup

First, we install **torch_brain**, and get set up with some simple utility functions that will be used throughout this notebook.

In [1]:
! uv pip install pytorch_brain -q

### Run the block below to load utility functions.

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import warnings
import logging
from torch_brain.utils import seed_everything

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)


def move_to_gpu(data, device):
    """
    Recursively moves tensors (or collections of tensors) to the given device.
    """
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, dict):
        return {k: move_to_gpu(v, device) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_gpu(elem, device) for elem in data]
    else:
        return data


def bin_spikes(spikes, num_units, bin_size, right=True, num_bins=None):
    """
    Bins spike timestamps into a 2D array: [num_units x num_bins].
    """
    rate = 1 / bin_size  # avoid precision issues
    binned_spikes = np.zeros((num_units, num_bins))
    bin_index = np.floor((spikes.timestamps) * rate).astype(int)
    np.add.at(binned_spikes, (spikes.unit_index, bin_index), 1)
    return binned_spikes


def r2_score(y_pred, y_true):
    # Compute total sum of squares (variance of the true values)
    y_true_mean = torch.mean(y_true, dim=0, keepdim=True)
    ss_total = torch.sum((y_true - y_true_mean) ** 2)

    # Compute residual sum of squares
    ss_res = torch.sum((y_true - y_pred) ** 2)

    # Compute R^2
    r2 = 1 - ss_res / ss_total

    return r2


def generate_sinusoidal_position_embs(num_timesteps, dim):
    position = torch.arange(num_timesteps).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2) * (-np.log(10000.0) / dim))
    pe = torch.empty(num_timesteps, dim)
    pe[:, 0:dim // 2] = torch.sin(position * div_term)
    pe[:, dim//2:] = torch.cos(position * div_term)
    return pe


def get_dataset_config(brainset, sessions):
    brainset_norms = {
        "perich_miller_population_2018": {
            "mean": 0.0,
            "std": 20.0
        }
    }

    config = f"""
    - selection:
      - brainset: {brainset}
        sessions:"""
    if type(sessions) is not list:
        sessions = [sessions]
    for session in sessions:
        config += f"""
          - {session}"""
    config += f"""
      config:
        readout:
          readout_id: cursor_velocity_2d
          normalize_mean: {brainset_norms[brainset]["mean"]}
          normalize_std: {brainset_norms[brainset]["std"]}
          metrics:
            - metric:
                _target_: torchmetrics.R2Score
    """

    config = OmegaConf.create(config)

    return config

/usr/local/lib/python3.12/dist-packages/temporaldata/temporaldata.py:1209: SyntaxWarning: invalid escape sequence '\*'
  multi-dimensional (2d, 3d, ..., nd) arrays with shape (N, \*).
/usr/local/lib/python3.12/dist-packages/einops/einops.py:737: SyntaxWarning: invalid escape sequence '\s'
  \sum_{c, d, g} x[a, b, c] * y[c, b, d] * z[a, g, k]


***

## Part 1: Data Loading

***

In [3]:
import torch

# Check GPU memory to help determine batch size capacity
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {device_name}")
    print(f"Total VRAM: {total_memory:.2f} GB")
    print("\nTip: If you encounter 'Out of Memory' errors, try halving your batch_size (e.g., from 64 to 32).")
else:
    print("No GPU detected. Use 'Runtime' > 'Change runtime type' and select a GPU accelerator.")

GPU: NVIDIA A100-SXM4-80GB
Total VRAM: 85.09 GB

Tip: If you encounter 'Out of Memory' errors, try halving your batch_size (e.g., from 64 to 32).


### Setting up a basic data pipeline

We define a utility function to create the training and validation datasets, samplers, and dataloaders.
This function will be used to set up the data for training and validation of the models.

In [4]:
from torch_brain.data import Dataset, collate, chain
from torch_brain.data.sampler import RandomFixedWindowSampler, SequentialFixedWindowSampler
from torch.utils.data import DataLoader

def get_train_val_loaders(recording_id=None, cfg=None, batch_size=32, seed=0):
    """Sets up train and validation Datasets, Samplers, and DataLoaders
    """

    # -- Train --
    train_dataset = Dataset(
        root="data",                # root directory where .h5 files are found
        recording_id=recording_id,  # you either specify a single recording ID
        config=cfg,                 # or a config for multi-session training / more complex configs
        split="train",
    )
    # We use a random sampler to improve generalization during training
    train_sampling_intervals = train_dataset.get_sampling_intervals()
    train_sampler = RandomFixedWindowSampler(
        sampling_intervals=train_sampling_intervals,
        window_length=1.0,          # context window of samples
        generator=torch.Generator().manual_seed(seed),
    )
    # Finally combine them in a dataloader
    train_loader = DataLoader(
        dataset=train_dataset,      # dataset
        sampler=train_sampler,      # sampler
        batch_size=batch_size,      # num of samples per batch
        collate_fn=collate,         # the collator
        num_workers=4,              # data sample processing (slicing, transforms, tokenization) happens in parallel; this sets the amount of that parallelization
        pin_memory=True,
    )

    # -- Validation --
    val_dataset = Dataset(
        root="data",
        recording_id=recording_id,
        config=cfg,
        split="valid",
    )
    # For validation we don't randomize samples for reproducibility
    val_sampling_intervals = val_dataset.get_sampling_intervals()
    val_sampler = SequentialFixedWindowSampler(
        sampling_intervals=val_sampling_intervals,
        window_length=1.0,
    )
    # Combine them in a dataloader
    val_loader = DataLoader(
        dataset=val_dataset,
        sampler=val_sampler,
        batch_size=batch_size,
        collate_fn=collate,
        num_workers=4,
        pin_memory=True,
    )

    train_dataset.disable_data_leakage_check()
    val_dataset.disable_data_leakage_check()

    return train_dataset, train_loader, val_dataset, val_loader

***

## Analysis

Paper Implementation:
1. Train Wiener Filter, GRU, MLP, and POYO on Day 1: `t_20130819_center_out_reaching`.
2. Fine-tune/adapt on the Day 2 train split: `t_20130821_center_out_reaching`.
3. Test on the held-out Day 2 validation split.

Independent Study:
Compare performance between POYO's single-spike event tokenization and binned spike-count baselines. The binned models use a shared global unit vocabulary so Day 1 and Day 2 produce the same feature columns.


### Architectural Comparison: Why Binned vs. Event-based?

| Model Type | Input Requirement | Why it needs Binning |
| :--- | :--- | :--- |
| **Linear/Ridge** | Fixed Vector | Requires a static feature matrix $X$; cannot handle variable-length event lists. |
| **MLP** | Fixed Tensor | Weights are mapped to specific indices; it needs to know which 'pixel' (neuron/time) it is looking at. |
| **GRU/RNN** | Clocked Sequences | Processes data in regular intervals; event-based triggers lead to vanishing gradients over long spike sequences. |
| **POYO** | Set/Point Cloud | Uses **Attention** to dynamically weigh events based on their continuous timestamps and learned unit embeddings. |

In [5]:
# Final project configuration
import torch.nn as nn
import torch.nn.functional as F
from torch_brain.nn import FeedForward

# Increase these for final numbers. The defaults are intentionally small enough for a quick smoke test.
DAY1_RECORDING_ID = "perich_miller_population_2018/t_20130819_center_out_reaching"
DAY2_RECORDING_ID = "perich_miller_population_2018/t_20130821_center_out_reaching"

BIN_SIZE = 10e-3
SEQUENCE_LENGTH = 1.0
BATCH_SIZE = 64

PRETRAIN_EPOCHS = 100
ADAPT_EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

WIENER_LAGS = 10
RIDGE_ALPHA = 1.0

results = []


In [6]:
# Download both sessions used in the final-project experiment.
! mkdir -p data/perich_miller_population_2018
! gdown 1W--Sm_BcphEC2snoF4zwPdHkkYGgAaUw -O data/perich_miller_population_2018/t_20130819_center_out_reaching.h5
! gdown 1EZUkOE8oiieWja9lblokf8WjF05HFuX- -O data/perich_miller_population_2018/t_20130821_center_out_reaching.h5


Downloading...
From: https://drive.google.com/uc?id=1W--Sm_BcphEC2snoF4zwPdHkkYGgAaUw
To: /content/data/perich_miller_population_2018/t_20130819_center_out_reaching.h5
100% 9.88M/9.88M [00:00<00:00, 31.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EZUkOE8oiieWja9lblokf8WjF05HFuX-
To: /content/data/perich_miller_population_2018/t_20130821_center_out_reaching.h5
100% 10.4M/10.4M [00:00<00:00, 27.5MB/s]


In [7]:
def unique_preserve_order(values):
    seen = set()
    out = []
    for value in values:
        value = str(value)
        if value not in seen:
            seen.add(value)
            out.append(value)
    return out


def make_loaders_for_recording(recording_id, batch_size=BATCH_SIZE, seed=0, transform=None):
    train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(
        cfg=get_dataset_config("perich_miller_population_2018", recording_id.split("/")[-1]),
        batch_size=batch_size,
        seed=seed,
    )
    if transform is not None:
        train_dataset.transform = transform
        val_dataset.transform = transform
    return train_dataset, train_loader, val_dataset, val_loader


# Build datasets once to discover stable unit/session vocabularies.
day1_train_ds, _, day1_valid_ds, _ = make_loaders_for_recording(DAY1_RECORDING_ID)
day2_train_ds, _, day2_test_ds, _ = make_loaders_for_recording(DAY2_RECORDING_ID)

GLOBAL_UNIT_IDS = unique_preserve_order(day1_train_ds.get_unit_ids() + day2_train_ds.get_unit_ids())
GLOBAL_SESSION_IDS = unique_preserve_order(day1_train_ds.get_session_ids() + day2_train_ds.get_session_ids())
GLOBAL_UNIT_TO_INDEX = {unit_id: i for i, unit_id in enumerate(GLOBAL_UNIT_IDS)}

print(f"Day 1 units: {len(day1_train_ds.get_unit_ids())}")
print(f"Day 2 units: {len(day2_train_ds.get_unit_ids())}")
print(f"Global binned feature columns: {len(GLOBAL_UNIT_IDS)}")
print(f"Sessions: {GLOBAL_SESSION_IDS}")


Day 1 units: 55
Day 2 units: 59
Global binned feature columns: 114
Sessions: ['perich_miller_population_2018/t_20130819_center_out_reaching', 'perich_miller_population_2018/t_20130821_center_out_reaching']


In [8]:
class GlobalBinnedTokenizer:
    """Converts a sliced torch_brain sample into fixed-width binned spike-count features."""

    def __init__(self, global_unit_ids, bin_size=BIN_SIZE, sequence_length=SEQUENCE_LENGTH):
        self.global_unit_ids = [str(unit_id) for unit_id in global_unit_ids]
        self.global_unit_to_index = {unit_id: i for i, unit_id in enumerate(self.global_unit_ids)}
        self.bin_size = bin_size
        self.num_timesteps = int(sequence_length / bin_size)

    def __call__(self, data):
        local_unit_ids = [str(unit_id) for unit_id in data.units.id]
        local_to_global = np.array([self.global_unit_to_index[unit_id] for unit_id in local_unit_ids])

        x_local = bin_spikes(
            spikes=data.spikes,
            num_units=len(local_unit_ids),
            bin_size=self.bin_size,
            num_bins=self.num_timesteps,
        ).T

        x = np.zeros((self.num_timesteps, len(self.global_unit_ids)), dtype=np.float32)
        x[:, local_to_global] = x_local.astype(np.float32)

        y = np.asarray(data.cursor.vel, dtype=np.float32)
        if y.shape[0] != self.num_timesteps:
            y = y[:self.num_timesteps]
            x = x[:len(y)]

        return {
            "model_inputs": {"x": torch.tensor(x, dtype=torch.float32)},
            "target_values": torch.tensor(y, dtype=torch.float32),
        }


binned_tokenizer = GlobalBinnedTokenizer(GLOBAL_UNIT_IDS, bin_size=BIN_SIZE, sequence_length=SEQUENCE_LENGTH)

day1_train_ds, day1_train_loader, _, _ = make_loaders_for_recording(DAY1_RECORDING_ID, transform=binned_tokenizer)
_, _, day2_test_ds, day2_test_loader = make_loaders_for_recording(DAY2_RECORDING_ID, transform=binned_tokenizer)
day2_train_ds, day2_adapt_loader, _, _ = make_loaders_for_recording(DAY2_RECORDING_ID, transform=binned_tokenizer)

NUM_GLOBAL_UNITS = len(GLOBAL_UNIT_IDS)
print(f"Binned loaders ready with {NUM_GLOBAL_UNITS} aligned unit columns.")


Binned loaders ready with 114 aligned unit columns.


In [9]:
def evaluate_decoder(model, dataloader):
    model.eval()
    total_target = []
    total_pred = []
    with torch.no_grad():
        for batch in dataloader:
            batch = move_to_gpu(batch, device)
            pred = model(**batch["model_inputs"])
            target = batch["target_values"]
            mask = torch.ones_like(target, dtype=torch.bool)
            if "output_mask" in batch["model_inputs"]:
                mask = batch["model_inputs"]["output_mask"]
            total_target.append(target[mask])
            total_pred.append(pred[mask])
    total_target = torch.cat(total_target)
    total_pred = torch.cat(total_pred)
    return r2_score(total_pred.flatten(), total_target.flatten()).item()


def fit_torch_decoder(model, train_loader, eval_loader, epochs, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_log = []
    r2_log = []

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = move_to_gpu(batch, device)
            optimizer.zero_grad()
            pred = model(**batch["model_inputs"])
            loss = F.mse_loss(pred, batch["target_values"])
            loss.backward()
            optimizer.step()
            loss_log.append(loss.item())

        r2 = evaluate_decoder(model, eval_loader)
        r2_log.append(r2)
        print(f"\rEpoch {epoch + 1}/{epochs} | Day 2 test R2 = {r2:.3f} | Loss = {loss.item():.4f}", end="")

    print()
    return r2_log, loss_log


def run_binned_torch_experiment(model_name, model):
    print(f"\n=== {model_name}: train Day 1, adapt Day 2, test held-out Day 2 ===")
    pre_r2_log, pre_loss_log = fit_torch_decoder(model, day1_train_loader, day2_test_loader, PRETRAIN_EPOCHS)
    r2_after_day1 = evaluate_decoder(model, day2_test_loader)

    adapt_r2_log, adapt_loss_log = fit_torch_decoder(model, day2_adapt_loader, day2_test_loader, ADAPT_EPOCHS)
    r2_after_adapt = evaluate_decoder(model, day2_test_loader)

    result = {
        "model": model_name,
        "New Day R^2": r2_after_adapt,
    }
    results.append(result)
    return result, pre_r2_log, adapt_r2_log, pre_loss_log + adapt_loss_log


### Binned Baselines

These models all consume the same aligned binned spike-count tensor: `[batch, time, global_units]`.


In [10]:
class FinalProjectMLPDecoder(nn.Module):
    def __init__(self, num_units, num_timesteps, output_dim=2, hidden_dim=128):
        super().__init__()
        self.num_timesteps = num_timesteps
        self.output_dim = output_dim
        self.net = nn.Sequential(
            nn.Linear(num_timesteps * num_units, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_timesteps * output_dim),
        )

    def forward(self, x):
        x = x.flatten(1)
        x = self.net(x)
        return x.reshape(-1, self.num_timesteps, self.output_dim)


class GRUNeuralDecoder(nn.Module):
    def __init__(self, num_units, hidden_dim=128, num_layers=1, output_dim=2, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=num_units,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.readout = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h, _ = self.gru(x)
        return self.readout(h)

In [12]:
# Run binned neural baselines.
# Comment out models here if you only want a subset during debugging.
num_timesteps = int(SEQUENCE_LENGTH / BIN_SIZE)

mlp_result, mlp_pre_r2, mlp_adapt_r2, mlp_loss = run_binned_torch_experiment(
    "MLP", FinalProjectMLPDecoder(NUM_GLOBAL_UNITS, num_timesteps, hidden_dim=128)
)

gru_result, gru_pre_r2, gru_adapt_r2, gru_loss = run_binned_torch_experiment(
    "GRU", GRUNeuralDecoder(NUM_GLOBAL_UNITS, hidden_dim=128, num_layers=1)
)



=== MLP: train Day 1, adapt Day 2, test held-out Day 2 ===
Epoch 1/100 | Day 2 test R2 = -0.001 | Loss = 17.3547

KeyboardInterrupt: 

### Wiener Filter Baseline

The Wiener filter uses lagged binned spike counts and Ridge regression. For adaptation, it refits on Day 1 plus Day 2 adaptation windows, then evaluates on held-out Day 2.


In [ ]:
from sklearn.linear_model import Ridge


def collect_binned_xy(dataloader):
    xs = []
    ys = []
    for batch in dataloader:
        xs.append(batch["model_inputs"]["x"].numpy())
        ys.append(batch["target_values"].numpy())
    x = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return x, y


def make_lagged_design(x, y, num_lags=WIENER_LAGS):
    # x: [windows, time, units], y: [windows, time, 2]
    features = []
    targets = []
    for x_window, y_window in zip(x, y):
        for t in range(num_lags - 1, x_window.shape[0]):
            features.append(x_window[t - num_lags + 1:t + 1].reshape(-1))
            targets.append(y_window[t])
    return np.asarray(features, dtype=np.float32), np.asarray(targets, dtype=np.float32)


def evaluate_wiener(model, dataloader, num_lags=WIENER_LAGS):
    x, y = collect_binned_xy(dataloader)
    x_design, y_target = make_lagged_design(x, y, num_lags=num_lags)
    y_pred = model.predict(x_design)
    y_pred_t = torch.tensor(y_pred, dtype=torch.float32)
    y_target_t = torch.tensor(y_target, dtype=torch.float32)
    return r2_score(y_pred_t.flatten(), y_target_t.flatten()).item()


print("\n=== Wiener filter: train Day 1, adapt Day 2, test held-out Day 2 ===")
x_day1, y_day1 = collect_binned_xy(day1_train_loader)
x_day2_adapt, y_day2_adapt = collect_binned_xy(day2_adapt_loader)

x_day1_design, y_day1_target = make_lagged_design(x_day1, y_day1, num_lags=WIENER_LAGS)
wiener = Ridge(alpha=RIDGE_ALPHA)
wiener.fit(x_day1_design, y_day1_target)
r2_after_day1 = evaluate_wiener(wiener, day2_test_loader)
print(f"Day 2 test R2 after Day 1 train: {r2_after_day1:.3f}")

x_day2_design, y_day2_target = make_lagged_design(x_day2_adapt, y_day2_adapt, num_lags=WIENER_LAGS)
x_adapt_design = np.concatenate([x_day1_design, x_day2_design], axis=0)
y_adapt_target = np.concatenate([y_day1_target, y_day2_target], axis=0)
wiener_adapted = Ridge(alpha=RIDGE_ALPHA)
wiener_adapted.fit(x_adapt_design, y_adapt_target)
r2_after_adapt = evaluate_wiener(wiener_adapted, day2_test_loader)
print(f"Day 2 test R2 after Day 2 adaptation: {r2_after_adapt:.3f}")

results.append({
    "model": "Wiener/Ridge",
    "New Day R^2": r2_after_adapt,
})


### POYO-Style Model With Binned Inputs

This is not the official POYO tokenizer. It is a POYO-inspired binned-input model: binned spike counts are read into time tokens, learned latents cross-attend to those tokens, then output time queries decode velocity. This gives a direct comparison to ask whether POYO-like latent attention still helps when the single-spike tokenization is removed.


In [ ]:
class BinnedPOYOStyleDecoder(nn.Module):
    def __init__(
        self,
        num_units,
        num_timesteps,
        dim=128,
        num_latents=128,
        depth=4,
        n_heads=4,
        output_dim=2,
    ):
        super().__init__()
        self.num_timesteps = num_timesteps
        self.readin = nn.Linear(num_units, dim)
        self.input_pos = nn.Parameter(generate_sinusoidal_position_embs(num_timesteps, dim), requires_grad=False)
        self.latents = nn.Parameter(torch.randn(num_latents, dim) * 0.02)
        self.output_queries = nn.Parameter(generate_sinusoidal_position_embs(num_timesteps, dim), requires_grad=True)

        self.input_cross_attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.blocks = nn.ModuleList([
            nn.ModuleList([
                nn.MultiheadAttention(dim, n_heads, batch_first=True),
                FeedForward(dim=dim),
            ])
            for _ in range(depth)
        ])
        self.output_cross_attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.readout = nn.Linear(dim, output_dim)

    def forward(self, x):
        batch_size = x.shape[0]
        tokens = self.readin(x) + self.input_pos[None, ...].to(x.device)
        latents = self.latents[None, ...].expand(batch_size, -1, -1)
        latents = latents + self.input_cross_attn(latents, tokens, tokens, need_weights=False)[0]

        for attn, ffn in self.blocks:
            latents = latents + attn(latents, latents, latents, need_weights=False)[0]
            latents = latents + ffn(latents)

        queries = self.output_queries[None, ...].expand(batch_size, -1, -1).to(x.device)
        decoded = queries + self.output_cross_attn(queries, latents, latents, need_weights=False)[0]
        return self.readout(decoded)


binned_poyo_result, binned_poyo_pre_r2, binned_poyo_adapt_r2, binned_poyo_loss = run_binned_torch_experiment(
    "Binned POYO-style", BinnedPOYOStyleDecoder(NUM_GLOBAL_UNITS, num_timesteps, dim=128, num_latents=128, depth=4, n_heads=4)
)


### Official POYO Event-Token Baseline

This model uses POYO's standard single-spike event tokenizer. Its unit and session vocabularies are initialized with both days so Day 2 units can be adapted without changing the model shape after Day 1 training.


In [ ]:
from torch_brain.models import POYO
from torch_brain.registry import MODALITY_REGISTRY


def make_poyo_loaders(recording_id, transform=None):
    train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(
        cfg=get_dataset_config("perich_miller_population_2018", recording_id.split("/")[-1]),
        batch_size=BATCH_SIZE,
        seed=0,
    )
    train_dataset.transform = transform
    val_dataset.transform = transform
    return train_dataset, train_loader, val_dataset, val_loader


def build_small_poyo(unit_ids, session_ids):
    model = POYO(
        sequence_length=SEQUENCE_LENGTH,
        readout_spec=MODALITY_REGISTRY['cursor_velocity_2d'],
        latent_step=1.0 / 8,
        num_latents_per_step=16,
        dim=64,
        depth=6,
        dim_head=64,
        cross_heads=2,
        self_heads=8,
    ).to(device)
    model.unit_emb.initialize_vocab(unit_ids)
    model.session_emb.initialize_vocab(session_ids)
    return model


print("\n=== Official POYO: train Day 1, adapt Day 2, test held-out Day 2 ===")
poyo_event_model = build_small_poyo(GLOBAL_UNIT_IDS, GLOBAL_SESSION_IDS)

poyo_day1_train_ds, poyo_day1_train_loader, _, _ = make_poyo_loaders(DAY1_RECORDING_ID, transform=poyo_event_model.tokenize)
poyo_day2_train_ds, poyo_day2_adapt_loader, _, _ = make_poyo_loaders(DAY2_RECORDING_ID, transform=poyo_event_model.tokenize)
_, _, poyo_day2_test_ds, poyo_day2_test_loader = make_poyo_loaders(DAY2_RECORDING_ID, transform=poyo_event_model.tokenize)

poyo_pre_r2, poyo_pre_loss = fit_torch_decoder(
    poyo_event_model,
    poyo_day1_train_loader,
    poyo_day2_test_loader,
    PRETRAIN_EPOCHS,
)
r2_after_day1 = evaluate_decoder(poyo_event_model, poyo_day2_test_loader)

poyo_adapt_r2, poyo_adapt_loss = fit_torch_decoder(
    poyo_event_model,
    poyo_day2_adapt_loader,
    poyo_day2_test_loader,
    ADAPT_EPOCHS,
)
r2_after_adapt = evaluate_decoder(poyo_event_model, poyo_day2_test_loader)

results.append({
    "model": "POYO",
    "New Day R^2": r2_after_adapt,
})


### Results Table

Use this table for the report/poster. The most important column is the held-out Day 2 R² after adaptation.


In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df = results_df[[
    "model",
    "New Day R^2",
]]
results_df.sort_values("New Day R^2", ascending=False)
